# Experiment: Gesture Classification with Deep Learning

In this experiment, a neural network model is trained with EMG data from the 65HDsEMG database to classify gestures. Here, a subset of the EMG channels is used to have the configuration/organization of that of the armband developed for this project. The data are divided into training and validation data to train the model, and into test data to evaluate the classification performance of the trained model.

# Signal preprocessing

The folowing 4-step preprocessing is applied to the EMG signal data:
1. Rectification
2. Teager-Kaiser Energy Operator (TKE)
3. Moving Average (30 samples)
4. Normalization

The normalization step is applied during the training while the other ones have already been applied to the data. 

## Table of Contents

- [0. Import libraries](#0)
- [1. Config Variables and hyperparameters](#1)
- [2. Load dataset with dataloader](#2)
- [3. Create model to train](#3)
- [4. Training](#4)
- [5. Evaluate best model](#5)
- [6. Save the model in .H5 file](#6)
- [7. Save the experiment in mlflow](#7)


<a name="0"></a>
# Import Libraries

In [1]:
import os
import sys

# Obtém o caminho da pasta atual
current_path  = os.getcwd()

In [2]:
import os, sys

import os, sys

processing_source_path = os.path.abspath('./../../../Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
import ProcessingPipeline as pp

dataloader_source_path = os.path.abspath('./../../../Dataloader/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
from EMGGestureDataGen import EMGGestureDataGen

dataloader_source_path = os.path.abspath('./../../../Dataloader/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
from EMGGestureDataGen import EMGGestureDataGen
from KerasDataPipeline import KerasDataPipeline

model_source_path = os.path.abspath('./../../../Keras_deep_models/')
if(model_source_path not in sys.path):
    sys.path.append(model_source_path)
from EMGConv2DResNetModel import EMGConv2DResNetModel

dataloader_source_path = os.path.abspath('./../../../Utils/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
import utils as utils
import available_metrics as metrics

Embedded_source_path = os.path.abspath('./../../../../embedded/Quantization/functions/')
if(Embedded_source_path not in sys.path):
    sys.path.append(Embedded_source_path)

from Embedded_Model import Embedded_Model
from Embedded_Model_Convertion import Embedded_Model_Convertion

In [3]:
import numpy as np
from os.path import join
import datetime
import matplotlib.pyplot as plt
import pandas as pd
import json

In [4]:
import tensorflow as tf
print(tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    # Currently, memory growth needs to be the same across GPUs
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Memory growth must be set before GPUs have been initialized
    print(e)

2.8.0
2 Physical GPUs, 2 Logical GPUs


2024-05-22 17:39:08.411980: I tensorflow/core/platform/cpu_feature_guard.cc:152] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE3 SSE4.1 SSE4.2 AVX
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-22 17:39:09.330053: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 472 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:0f:00.0, compute capability: 8.0
2024-05-22 17:39:09.331576: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 77655 MB memory:  -> device: 1, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:47:00.0, compute capability: 8.0


<a name="1"></a>
# Config Variables and hyperparameters

In [5]:
# model_name = 'EMG65_2D_CNN_multiclass_armband_onesubject20230627_194533'
model_name = None

In [11]:
# Parameters
root_path='./../../../../../../'

dataset_path = "/root/AI-Algoritmhs-Main/EMG/DataBase_Treatment/ELD_create_databaseH5/record_examples/"
h5_file_path_train = ""
h5_file_path_val = ""
h5_file_path_test = ""
channels = ""
sampling_frequency = 2034.179688
extracted_frequencies = [10, 500]
num_emg_channels = 64
features_type = ["raw"]
preprocessing = ["HighPassFilter", "LowPassFilter"]


Grip_Patterns = ["Rest", "Pinch", "Tripod_Grip", "Power_Grip", "Hook_Grip", "Open_Hand", "Point"]
subject_name = "Tiago_Dezotti"

#-----------------------------------------------------------------------------------------------------
#Grip_Patterns = ["Rest", "Power_Grip","Open_Hand"]
num_classes = len(Grip_Patterns)

subject_name = "sub03"
subject =0

In [12]:
if model_name is None:
    print('Crate new config model')
    data_params = {
        "checkpoint_path": "trainings/",
        "training_name": 'EMG65_2D_CNN_multiclass_armband' + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"),
        "subject": subject,
        'h5_file_name': 'All',
        "dataset_path": os.path.join(dataset_path, f'{subject}.{subject_name}', 'ARMBAND'),
        "dataset_name": 'ELD_armband_HD-EMG',
        "Grip_Patterns": Grip_Patterns,
        "num_classes" : num_classes,
        "num_emg_channels": 64,
        "channels": 64,
        'samples_per_batch':256,
        'sampling_frequency':sampling_frequency,
        'Used_channels_matrix':[]
    }
    training_params = {
        "using_armband": True,
        "GPU": True,
        "early_stopping_patient": 15,
        "epochs": 200,
        "batch_size": 32,
        "lr": 0.01,
        "loss": 'CatCE',
        
        "filter_divisor": 4,
        "window_size": 2**10, # window size- num fetures
        "window_step": 2**5,  # window step
    }
    model_params = {
        'IA_Technique': 'deep-learning', 
        "model_output_mode": 'multiclass',
        "Models_name": ['Resnet2D_all'],
        "model_output_mode": 'multiclass',
        "print_model": True,
        "network_name": 'Resnet2D_all',
        "optimizer": 'adamW',
    }

    preprocessing_params = {
    'norm_axis': 2, # 1 is good for acc but is no robust; 2 is more robust but lower acc
    
    "img_width": 4, #8
    "img_height": 8, #
    "n_voting": 7,
    'preprocessing': ['HighPassFilter','LowPassFilter'],
    'extracted_frequencies': extracted_frequencies,
    }
    preprocessing_params['HighPassFilter']= preprocessing_params['extracted_frequencies'][0]
    preprocessing_params['LowPassFilter']= preprocessing_params['extracted_frequencies'][1]

    # Combine all categories into a single dictionary
    params = {**data_params, **model_params, **training_params, **preprocessing_params}
    
else:
    config_path = join('trainings/', model_name, 'training_param.json')
    print('Loading config model from:', config_path)
    with open(config_path) as json_file:
        params = json.load(json_file)
        
params['num_features']=params['window_size']
params['h5_file_path_train'] = os.path.join(params['dataset_path'],'train')
params['h5_file_path_val'] = os.path.join(params['dataset_path'],'val')
params['h5_file_path_test'] = os.path.join(params['dataset_path'],'test')
print(params)

Crate new config model
{'checkpoint_path': 'trainings/', 'training_name': 'EMG65_2D_CNN_multiclass_armband20240522_174402', 'subject': 0, 'h5_file_name': 'All', 'dataset_path': '/root/AI-Algoritmhs-Main/EMG/DataBase_Treatment/ELD_create_databaseH5/record_examples/0.sub03/ARMBAND', 'dataset_name': 'ELD_armband_HD-EMG', 'Grip_Patterns': ['Rest', 'Pinch', 'Tripod_Grip', 'Power_Grip', 'Hook_Grip', 'Open_Hand', 'Point'], 'num_classes': 7, 'num_emg_channels': 64, 'channels': 64, 'samples_per_batch': 256, 'sampling_frequency': 2034.179688, 'Used_channels_matrix': [], 'IA_Technique': 'deep-learning', 'model_output_mode': 'multiclass', 'Models_name': ['Resnet2D_all'], 'print_model': True, 'network_name': 'Resnet2D_all', 'optimizer': 'adamW', 'using_armband': True, 'GPU': True, 'early_stopping_patient': 15, 'epochs': 200, 'batch_size': 32, 'lr': 0.01, 'loss': 'CatCE', 'filter_divisor': 4, 'window_size': 1024, 'window_step': 32, 'norm_axis': 2, 'img_width': 4, 'img_height': 8, 'n_voting': 7, 'pre

In [13]:

#params['Used_channels_matrix']=np.array([13,  6, 11, 15,  7, 10, 58, 12,  4, 57, 14,  5, 16, 40, 23, 32, 17, 39])
#params['img_width']=2
#params['img_height']=9


In [14]:
os.path.join(params['dataset_path'], 'train',f'{subject_name}.hdf5')

'/root/AI-Algoritmhs-Main/EMG/DataBase_Treatment/ELD_create_databaseH5/record_examples/0.sub03/ARMBAND/train/sub03.hdf5'

In [15]:
import h5py

with h5py.File(os.path.join(params['dataset_path'], 'train',f'{subject_name}.hdf5')) as hf:
        #params['sessions']=int(np.max(hf['sessions'][:]))
        #print('Number of recorded sessions:',params['sessions'])
        print(hf.keys())
        sessions=hf['sessions'][:]
        print(hf[f'session_{sessions[0]}'].keys)
        print('gesture_0:',hf[f'session_{sessions[0]}']['gesture_0'].keys())

        #emg=hf['session_1']['gesture_1']['repetition_1']['emg'][...]
del hf

OSError: Unable to open file (truncated file: eof = 268435456, sblock->base_addr = 0, stored_eof = 290092848)

<a name="2"></a>
# Load dataset with dataloader

In [ ]:
params['h5_file_path_train']

In [ ]:
data_processing = pp.ProcessingPipeline([
    pp.NormMinMax(axis=params['norm_axis']),
    pp.DataReshape((params['img_width'], params['img_height'], params['window_size'])),
    pp.LabelOneHot(np.sort(np.arange(len(params['Grip_Patterns'])))) # to multiclass 
    ])

train_dataset = EMGGestureDataGen(params['h5_file_path_train'], params['batch_size'], params['window_size'], params['window_step'], params['channels'], processing=data_processing, shuffle=True, armband=params['using_armband'], dataset_name='ELD', subjects=params['h5_file_name'],Used_channels_matrix=params['Used_channels_matrix'])
val_dataset = EMGGestureDataGen(params['h5_file_path_val'], params['batch_size'], params['window_size'], params['window_step'], params['channels'], processing=data_processing, shuffle=True, armband=params['using_armband'], dataset_name='ELD', subjects=params['h5_file_name'],Used_channels_matrix=params['Used_channels_matrix'])
test_dataset = EMGGestureDataGen(params['h5_file_path_test'], params['batch_size'], params['window_size'], params['window_step'], params['channels'], processing=data_processing, shuffle=False, armband=params['using_armband'], dataset_name='ELD', subjects=params['h5_file_name'],Used_channels_matrix=params['Used_channels_matrix'])


In [ ]:
params['Used_channels_matrix'] = np.hstack(test_dataset.Used_channels_matrix).tolist()
print('(bach size, num_channels, window size')
test_dataset.__getitem__(0)[0].shape

In [ ]:
np.vstack(test_dataset.__getitem__(0)[0][0]).T.shape


In [ ]:
test_dataset.__getitem__(0)[0][0].shape

In [ ]:
plt.plot(np.vstack(test_dataset.__getitem__(0)[0][0]).T)
plt.title(f'1-batch of signals normalized label: {test_dataset.__getitem__(0)[1][0]}')
plt.xlabel('samples')
plt.ylabel('Normalized Amplitude')
plt.show()

plt.plot(np.hstack(test_dataset.__getitem__(20)[0][0]).T)
plt.title(f'1-batch of signals normalized label: {test_dataset.__getitem__(20)[1][0]}')
plt.xlabel('samples')
plt.ylabel('Normalized Amplitude')
plt.show()

In [ ]:
def graph_hd_extensor_matrix(data_matrix, ch_ref=32, tittle_label=''):
    """
    It creates heatmap plot.

    :param data_matrix: Matrix with data channels
    :param muscle: Choosing the muscle that will be used
    """
    num_electrodes_x = data_matrix.shape[1]
    num_electrodes_y = data_matrix.shape[0]
    fig_width = num_electrodes_x
    fig_height = num_electrodes_y
    ch_ref = 0
    plt.figure(figsize=(fig_width + 0, fig_height + 0))  # Tamanho da figura

    plt.imshow(data_matrix, cmap='Reds',
                extent=[0.0, float(num_electrodes_x), 0.0, float(num_electrodes_y)])

    ch_ref = ch_ref + 1

    for i in range(num_electrodes_y):
        for j in range(num_electrodes_x):
            plt.text(j + 0.5, i + 0.5,
                        str(ch_ref - (i * num_electrodes_x + (num_electrodes_x - j))),
                        color='black',
                        ha='center', va='center', fontsize=8)

    plt.xticks(np.arange(num_electrodes_x) + 0.5, range(1, num_electrodes_x + 1)) # type: ignore 
    plt.yticks(np.flip(np.arange(num_electrodes_y)) + 0.5, range(1, num_electrodes_y + 1)) # type: ignore
    plt.xlabel('Electrodes')
    plt.ylabel('Electrodes')

    plt.title(f'Electrode Heat Map per batch| label:{tittle_label}')
    plt.grid(False)

    plt.show()

for idx in np.random.randint(1,100,size=10):
    RMS=pp.DataRMS()
    data_rms,_=RMS(train_dataset[idx][0][0])
    print(data_rms.shape)
    graph_hd_extensor_matrix(data_rms, ch_ref=32, tittle_label=np.argmax(train_dataset[idx][1][0]))

In [ ]:
np.unique(test_dataset.yData,return_counts=True)

In [ ]:
# UNCOMMENT TO USE CLASS WEIGHTS
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight('balanced', classes=np.arange(len(params['Grip_Patterns'])), y=train_dataset.yData)
params['class_weights'] = dict(enumerate(class_weights))
class_weights

In [ ]:
train_dataset_pipeline = KerasDataPipeline(train_dataset, (None, params['img_width'], params['img_height'], params['window_size'] ), [None, params['num_classes']])
val_dataset_pipeline = KerasDataPipeline(val_dataset, (None, params['img_width'], params['img_height'], params['window_size']), [None, params['num_classes']])

<a name="3"></a>
# Create model to train

In [ ]:
emg_model = EMGConv2DResNetModel(params)
emg_model.load_model()

<a name="4"></a>
# Training

In [ ]:
history = emg_model.train(train_dataset_pipeline.datasetPipeline, val_dataset_pipeline.datasetPipeline)

with open(join(params['checkpoint_path'], params['network_name'], 'training_param.json'), 'w') as fp:
    json.dump(params, fp, indent=4)

In [ ]:
from pandas import read_csv
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.title("Loss")
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()

plt.figure(figsize=(15,5))
plt.title("Accuracy")
plt.plot(history.history['categorical_accuracy'], label='Train')
plt.plot(history.history['val_categorical_accuracy'], label='Validation')
plt.legend()

<a name="5"></a>
# Evaluate best model

In [ ]:
import tensorflow_addons as tfa
# load best model
trained_deep_model = tf.keras.models.load_model(join(params['checkpoint_path'] + params['network_name']))

<a name="5-1"></a>
### Train Dataset

In [ ]:
test_performance = trained_deep_model.evaluate(train_dataset)
y_train, y_pred = utils.generate_data(train_dataset, trained_deep_model)

In [ ]:
pr, re, f1, acc=metrics.model_metrics(params['Grip_Patterns'] ,y_train, y_pred, Get_metrics=True) 

In [ ]:
# available the train data 
cfm_deep_model=metrics.plot_confusion_matrix(params['Grip_Patterns'], y_train, y_pred, Get_matrix=True)

<a name="5-2"></a>
### Test Dataset

In [ ]:
test_performance = trained_deep_model.evaluate(test_dataset)
y_test, y_pred = utils.generate_data(test_dataset, trained_deep_model);

In [ ]:
pr, re, f1, acc=metrics.model_metrics(params['Grip_Patterns'] ,y_test, y_pred, Get_metrics=True) 

In [ ]:
utils.majority_voting(y_pred,2)

In [ ]:
y_test

In [ ]:
# available the test data 
cfm_deep_model=metrics.plot_confusion_matrix(params['Grip_Patterns'], y_test, y_pred, Get_matrix=True)
#cfm_deep_model=metrics.plot_confusion_matrix(params['Grip_Patterns'], y_test, utils.majority_voting(y_pred,params['n_voting']), Get_matrix=True)

#### Metrics with Average and std for each classes

In [ ]:
metrics.plot_metric_per_class(cfm_deep_model, params['Grip_Patterns'], metric_name='Accuracy', method='per_class')

In [ ]:
metrics.plot_metric_per_class(cfm_deep_model, params['Grip_Patterns'], metric_name='F1-Score', method='per_class')

#### Mean and std for all Classes 

In [ ]:
metrics_per_class,precision_list,recall_list,f1_score_list,accuracy_list=metrics.model_average_std_metrics(cf_matrix=cfm_deep_model, Get_metrics=True, Verbose=True) 

In [ ]:
metrics_average_and_std_df=pd.DataFrame(data=[precision_list,recall_list,f1_score_list,accuracy_list],
             index=['pr', 're', 'f1', 'acc'],
             columns=['Average','std'])
metrics_average_and_std_df

## ROC

In [ ]:
roc_score_per_class=metrics.get_roc_curve(params['Grip_Patterns'], y_test, y_pred ,verbose=True)

In [ ]:
# import mlflow_save_parameters_multiclass 

# mlflow_save_parameters_multiclass.save_data_in_mlflow("65HDsEMG_multiclass_armband", params, test_metric, data_processing.getProcessingNameList())

<a name="6"></a>
## Save the model in .H5 file

In [ ]:
params['window_step']=256

In [ ]:
subject=params['subject']

# save model's params
trained_deep_model.save(params['checkpoint_path'], f'/{params["Models_name"][0]}.h5')

In [ ]:
# save JSON 
json_filename=f'./../../../embedded/EMG/keras_quantized_model/model_{params["Models_name"][0]}_params.json'
with open(json_filename, 'w') as json_file:
    json.dump(params, json_file, indent=4)

print(f'Saving model params in JSON: {json_filename}')
#print(json_parameters)

* Get the quantized model:

In [ ]:
destination_path=f'./../../../embedded/EMG/keras_quantized_model/{params["Models_name"][0]}'
tflite_model=Embedded_Model_Convertion(os.path.join(params['checkpoint_path'],params["Models_name"][0]), destination_path)
destination_path

<a name="7"></a>
## Save Experiment in mlflow

In [ ]:
from mlflow.tracking import MlflowClient
import mlflow
import sys
import shutil
import os
experiment_name = 'ARMBAND_EMG_experiment'

root_path = os.path.join(root_path,'mlruns')

def save_in_mlflow(params, cfm=None, fig_pca=None, roc_score_per_class=None):
    # Set the tracking URI to a local directory
    sys.path.append(root_path)
    mlflow.set_tracking_uri(f"file:{root_path}")

    mlflow.set_experiment(experiment_name)
    experiment = mlflow.get_experiment_by_name(experiment_name)
    
    with mlflow.start_run(experiment_id=experiment.experiment_id):
        # Log data_params
        for category in params:
            mlflow.log_param(category, params[category])
        
        if roc_score_per_class is not None:
            # Log ROC values per class
            mlflow.log_param('ROC_AUCs', list(roc_score_per_class))
            
        if cfm is not None:
            # Log confusion matrix values
            mlflow.log_param("confusion_matrix", cfm.tolist())

        if os.path.exists(fig_pca):
            mlflow.log_artifact(fig_pca, artifact_path='images')
            os.remove(fig_pca)
            mlflow.log_param('features_pca_name', fig_pca)

In [ ]:
save_in_mlflow(params,cfm_deep_model,'3d_scatter_plot.png', roc_score_per_class)